# 02 - Per-pixel sampling through the Earth Engine high-volume endpoint

**Purpose** - extract the full monthly feature time series for every sampled pixel.
`ee.data.computePixels` is issued against the high-volume endpoint, which returns
pixel values directly into Python as NumPy arrays. At this sample size that is
substantially more reliable than queueing thousands of `Export.table.toDrive` tasks,
which run into rate limiting and intermittent 502 responses.

**Main inputs** - `data/raw/pixel_coords_10m.csv` (the 608,922 sampled pixels, whose
selection depends on the restricted fire-reference inventory) and the Earth Engine
assets built in notebook 01: the Sentinel-2 collection, the seasonal baselines, the
land-cover layer and the monthly fire labels.

**Main outputs** - `data/raw/full_pixels_10m.csv`, the per-pixel monthly series
carrying the 52 features per month, written incrementally through `partial_batches/`
so an interrupted run resumes.

**Role in the thesis workflow** - the bridge between Earth Engine and the modelling
stage. Notebook 04 reads its output to build the model-ready arrays.

**Requirements** - the Earth Engine project must have high-volume endpoint access.
Expect roughly 30 to 45 minutes on Colab and 15 to 20 minutes locally.

In [ ]:
# --- Environment auto-detection ---
import sys, os
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    THESIS_ROOT = "/content/drive/MyDrive/thesisv4"
else:
    THESIS_ROOT = os.path.expanduser("~/thesisv4")

sys.path.insert(0, THESIS_ROOT)
print(f"=== Environment: {'COLAB' if IS_COLAB else 'LOCAL'} ===")
print(f"Workspace: {THESIS_ROOT}")

In [ ]:
import importlib
import config
importlib.reload(config)
config.ensure_dirs()
config.set_seed()
config.print_summary()

In [ ]:
import ee
import io
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import time

# CRITICAL: Use the high-volume endpoint for parallel computePixels requests
HIGH_VOLUME_URL = "https://earthengine-highvolume.googleapis.com"

try:
    ee.Initialize(project=config.GEE["project"], opt_url=HIGH_VOLUME_URL)
    print("✓ Initialized against HIGH-VOLUME endpoint")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=config.GEE["project"], opt_url=HIGH_VOLUME_URL)
    print("✓ Authenticated and initialized against HIGH-VOLUME endpoint")

## 1. Load pixel coordinates and check inputs

In [ ]:
COORDS_PATH = config.FILES["pixel_coords"]
assert os.path.exists(COORDS_PATH), (
    f"Pixel coords not found: {COORDS_PATH}\n"
    "Run Steps 1 and 2 of `02_sampling.ipynb` first to build this file."
)

df_coords = pd.read_csv(COORDS_PATH)
print(f"Total pixels: {len(df_coords):,}")
print(f"\npixel_type distribution:")
print(df_coords["pixel_type"].value_counts())
print(f"\nColumns: {df_coords.columns.tolist()}")

## 2. Build the per-month feature stack

For each month in the analysis period an `ee.Image` is assembled carrying all 52
features: the raw bands, the spectral indices, the anomalies against the seasonal
baseline, and the land-cover one-hot encodings. The image stays a deferred
server-side expression and is not computed here.

`computePixels` evaluates that expression on the server and returns only the values
at the sample coordinates.

In [ ]:
# Load the pre-built ImageCollections
ic_s2       = ee.ImageCollection(config.GEE["ic_s2"])
ic_labels   = ee.ImageCollection(config.GEE["ic_labels"])
baseline_m  = ee.ImageCollection(config.GEE["ic_baseline_mean"])
baseline_sd = ee.ImageCollection(config.GEE["ic_baseline_std"])
lc_yearly   = ee.ImageCollection(config.GEE["landcover_yearly"])

print(f"S2 collection size: {ic_s2.size().getInfo()}")
print(f"Labels size:        {ic_labels.size().getInfo()}")

In [ ]:
MONTHS = config.get_month_list()  # 108 (year, month) tuples
print(f"Months to process: {len(MONTHS)}")

# The median baseline is also needed: it is the reference for raw_anomaly
baseline_med = ee.ImageCollection(config.GEE["ic_baseline_median"])


def add_indices(img_ref):
    """Compute the 7 spectral indices from raw bands.

    Uses the same index formulas as the monthly composites built in notebook 01.
    """
    eps = ee.Image.constant(1e-6)

    b4  = img_ref.select("B4")
    b8  = img_ref.select("B8")
    b8a = img_ref.select("B8A")
    b11 = img_ref.select("B11")
    b12 = img_ref.select("B12")

    ndvi = img_ref.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndmi = img_ref.normalizedDifference(["B8", "B11"]).rename("NDMI")
    nbr  = img_ref.normalizedDifference(["B8", "B12"]).rename("NBR")

    evi = (b8.subtract(b4).multiply(2.5)
             .divide(b8.add(b4.multiply(2.4)).add(1).add(eps))
             .rename("EVI"))

    mirbi = (b12.multiply(10).subtract(b11.multiply(9.8)).add(2)).rename("MIRBI")

    bsi = ((b11.add(b4).subtract(b8.add(b8a)))
           .divide(b11.add(b4).add(b8).add(b8a).add(eps))
           .rename("BSI"))

    term1 = ee.Image(1).subtract(
        b11.multiply(b12).multiply(b8a).divide(b4.add(eps)).sqrt())
    term2 = b12.subtract(b8a).divide(b12.add(b8a).add(eps).sqrt()).add(1)
    bais2 = term1.multiply(term2).rename("BAIS2")

    return (img_ref.select(config.RAW_BANDS)
                    .addBands([ndvi, evi, ndmi, nbr, mirbi, bsi, bais2])
                    .toFloat())


def get_baseline(ic, month, bands):
    return ee.Image(
        ic.filter(ee.Filter.eq("month", month)).first()
    ).select(bands)


def compute_anomaly_image(observed_with_indices, month):
    observed = observed_with_indices.select(config.ANOMALY_SOURCE_INDICES).unmask(-9999)

    b_mean   = get_baseline(baseline_m,   month, config.ANOMALY_SOURCE_INDICES)
    b_median = get_baseline(baseline_med, month, config.ANOMALY_SOURCE_INDICES)
    b_std    = get_baseline(baseline_sd,  month, config.ANOMALY_SOURCE_INDICES)

    eps = ee.Image.constant(1e-6)

    z_score = (observed
               .subtract(b_mean)
               .divide(b_std.add(eps))
               .rename([b + "_zscore" for b in config.ANOMALY_SOURCE_INDICES]))

    raw_anomaly = (observed
                   .subtract(b_median)
                   .rename([b + "_anomaly" for b in config.ANOMALY_SOURCE_INDICES]))

    return z_score.addBands(raw_anomaly)


def lc_year_index(year):
    return max(int(year), 2018)


def lc_image_for_year(year):
    idx = str(lc_year_index(year))
    return ee.Image(lc_yearly.filter(ee.Filter.eq("system:index", idx))
                        .first()).select("b1")


def lc_onehot_image(year, feature_names):
    lc_img = lc_image_for_year(year)
    bands = [lc_img.eq(code).toFloat().rename(name)
             for code, name in zip(config.LC_CODES, feature_names)]
    return ee.Image.cat(bands)


def build_month_image(year, month):
    """Build the feature stack ee.Image for a single month.

    Filters ImageCollections by (year, month) properties - not by system:index.
    Computes indices and anomalies on the fly from the raw bands.
    """
    s2_raw = (ic_s2.filter(ee.Filter.eq("year", year))
                    .filter(ee.Filter.eq("month", month))
                    .first().select(config.RAW_BANDS + ["pct_valid"]).unmask(-9999))

    s2_with_indices = add_indices(s2_raw)
    pct_valid_band  = s2_raw.select("pct_valid")

    anom_img = compute_anomaly_image(s2_with_indices, month).unmask(-9999)

    lbl_img = (ic_labels.filter(ee.Filter.eq("year", year))
                        .filter(ee.Filter.eq("month", month))
                        .first().unmask(0).rename("burned"))

    lc_prev_img = lc_onehot_image(year - 1, config.LC_FEATURES_PREV)
    lc_curr_img = lc_onehot_image(year,     config.LC_FEATURES_CURR)

    return (s2_with_indices
            .addBands(pct_valid_band)
            .addBands(anom_img)
            .addBands(lbl_img)
            .addBands(lc_prev_img)
            .addBands(lc_curr_img))


# Sanity check: build one image and print the band names
test_img = build_month_image(2020, 6)
band_names = test_img.bandNames().getInfo()
print(f"Bands per month image: {len(band_names)}")
print(f"Band names sample: {band_names[:10]} ... {band_names[-5:]}")

## 3. computePixels helper

For each (month, batch of pixels) request, we ask GEE to return the pixel
values at our sample coordinates as a NumPy structured array.

In [ ]:
# Split pixel_coords into batches. Keep each batch small enough that the
# response stays well under GEE's ~48 MB response cap.
#
# Response size ≈ n_pixels × n_bands × 4 bytes
#                = n_pixels × 46 × 4 = 184 * n_pixels bytes
# For a safe 30 MB cap → n_pixels ≤ ~160,000
# We use 50,000 to leave plenty of margin and reduce per-request latency.
PIXELS_PER_REQUEST = 20000  # kept modest to stay well under Colab RAM
n_batches = (len(df_coords) + PIXELS_PER_REQUEST - 1) // PIXELS_PER_REQUEST
print(f"Pixel batches per month: {n_batches} of ~{PIXELS_PER_REQUEST} pixels each")
print(f"Total requests: {len(MONTHS)} months × {n_batches} batches = "
      f"{len(MONTHS) * n_batches}")

In [ ]:
# We use `sampleRegions` server-side and fetch the result via
# `ee.data.computeFeatures` with a PANDAS_DATAFRAME output.
#
# To keep Python RAM low, we DON'T build local ee.Feature objects.
# Instead, we send the raw coordinates as a numpy array to ee.List
# and construct the FeatureCollection server-side inside a single
# ee.List.map. This keeps only the raw numbers in Python memory,
# not thousands of Python-side ee.Feature wrappers.

def make_batch_fc(df_batch):
    """Build an ee.FeatureCollection server-side from raw coords.

    Passes coords as a plain 2D list - GEE builds the Features on
    the server. Much lighter on client RAM than looping in Python.
    """
    coords = df_batch[["lon", "lat"]].values.tolist()
    pixel_ids = df_batch["pixel_id"].astype(int).tolist()

    # Zip [lon, lat, pixel_id] on the server
    combined = ee.List([
        [c[0], c[1], pid] for c, pid in zip(coords, pixel_ids)
    ])

    def to_feature(item):
        item = ee.List(item)
        lon = ee.Number(item.get(0))
        lat = ee.Number(item.get(1))
        pid = ee.Number(item.get(2))
        return ee.Feature(
            ee.Geometry.Point([lon, lat]),
            {"pixel_id": pid}
        )

    return ee.FeatureCollection(combined.map(to_feature))


def fetch_month_batch(year, month, df_batch, max_retries=5):
    """Fetch pixel values for a batch of coordinates in one month.

    Uses ee.data.computeFeatures to return a pandas DataFrame directly.
    Retries transient errors (502, quota exceeded) with exponential backoff.
    """
    img = build_month_image(year, month)
    fc  = make_batch_fc(df_batch)
    sampled = img.sampleRegions(
        collection=fc,
        scale=config.SCALE,
        projection=config.CRS,
        geometries=False,
        tileScale=4,
    )

    delay = 1.0
    for attempt in range(max_retries):
        try:
            df_out = ee.data.computeFeatures({
                "expression": sampled,
                "fileFormat": "PANDAS_DATAFRAME",
            })
            df_out["year"]  = year
            df_out["month"] = month
            return df_out
        except Exception as e:
            err = str(e).lower()
            if attempt == max_retries - 1:
                raise
            if "quota" in err or "rate" in err or "502" in err or "unavailable" in err:
                time.sleep(delay)
                delay = min(delay * 2, 30)
                continue
            raise


# Sanity check: fetch ONE small batch to verify the pipeline works
print("Sanity check: fetching 100 pixels for 2020-06...")
df_test = fetch_month_batch(2020, 6, df_coords.head(100))
print(f"  Got {len(df_test)} rows, {df_test.shape[1]} columns")
print(f"  Columns sample: {df_test.columns.tolist()[:8]}")
print(f"  First row NBR: {df_test['NBR'].iloc[0]}")
print("  ✓ Sanity check passed — pipeline works.")

## 4. Parallel fetch loop

All (month, batch) jobs are dispatched across a thread pool. The high-volume
endpoint sustains 40 or more concurrent requests, which brings the extraction down
from hours to minutes.

In [ ]:
# Prepare batch slices once
batches = []
for b in range(n_batches):
    lo = b * PIXELS_PER_REQUEST
    hi = min(lo + PIXELS_PER_REQUEST, len(df_coords))
    batches.append((b, df_coords.iloc[lo:hi].reset_index(drop=True)))

# Build the full list of (year, month, batch_id) jobs
all_jobs = []
for y, m in MONTHS:
    for b, _ in batches:
        all_jobs.append((y, m, b))

print(f"Total jobs (all 108 months): {len(all_jobs)}")

# ===================================================================
# TEST MODE - run one month only, verify output, then set to False
# ===================================================================
TEST_ONE_MONTH = False  # ← switched to full-run after single-month test passed
TEST_MONTH     = (2020, 6)

if TEST_ONE_MONTH:
    all_jobs = [(y, m, b) for (y, m, b) in all_jobs if (y, m) == TEST_MONTH]
    print(f"\n*** TEST MODE ON ***")
    print(f"Running only {TEST_MONTH[0]}-{TEST_MONTH[1]:02d} — {len(all_jobs)} jobs")
    print(f"After this succeeds, set TEST_ONE_MONTH=False and re-run.")
else:
    print(f"Full-run mode: {len(all_jobs)} jobs across all months")

print(f"Concurrent workers: 8 (safe for Colab RAM)")

In [ ]:
# Save partial results to disk so a crash mid-run doesn't cost hours.
PARTIAL_DIR = os.path.join(config.PATHS["data_raw"], "partial_batches")
os.makedirs(PARTIAL_DIR, exist_ok=True)


def partial_path(year, month, batch_id):
    return os.path.join(PARTIAL_DIR, f"m_{year}_{month:02d}_b_{batch_id:03d}.parquet")


def already_done(year, month, batch_id):
    p = partial_path(year, month, batch_id)
    return os.path.exists(p) and os.path.getsize(p) > 0


# Skip jobs that already have a saved partial (allows resuming after a crash)
jobs_to_run = [(y, m, b) for (y, m, b) in all_jobs
               if not already_done(y, m, b)]
print(f"Already done: {len(all_jobs) - len(jobs_to_run)}")
print(f"Remaining:    {len(jobs_to_run)}")

In [ ]:
def run_one_job(year, month, batch_id):
    """Fetch a (month, batch) and persist the resulting DataFrame as parquet."""
    _, df_batch = batches[batch_id]
    df_out = fetch_month_batch(year, month, df_batch)
    df_out.to_parquet(partial_path(year, month, batch_id), index=False)
    return (year, month, batch_id, len(df_out))


N_WORKERS = 16

errors = []
start_t = time.time()

with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = {ex.submit(run_one_job, y, m, b): (y, m, b)
               for (y, m, b) in jobs_to_run}

    pbar = tqdm(total=len(futures), desc="Fetching")
    for fut in as_completed(futures):
        y, m, b = futures[fut]
        try:
            _y, _m, _b, n_rows = fut.result()
            pbar.set_postfix_str(f"last: {_y}-{_m:02d} b{_b} n={n_rows}")
        except Exception as e:
            errors.append((y, m, b, str(e)))
            pbar.set_postfix_str(f"ERR {y}-{m:02d} b{b}")
        pbar.update(1)
    pbar.close()

elapsed = time.time() - start_t
print(f"\nFinished in {elapsed/60:.1f} minutes")
print(f"Errors: {len(errors)}")
for e in errors[:5]:
    print(f"  {e[0]}-{e[1]:02d} b{e[2]}: {e[3][:120]}")

## 5. Merge all partial files into `full_pixels_10m.csv`

In [ ]:
import os, glob

DRIVE_PARTIAL = "/content/drive/MyDrive/thesisv4/data/raw/partial_batches"

# List all files in Drive
all_files = sorted(glob.glob(f"{DRIVE_PARTIAL}/*.parquet"))
print(f"Total files in Drive: {len(all_files)}")

# Expected: 108 months × 31 batches = 3348
n_months = 108
n_batches = 31
expected = n_months * n_batches
print(f"Expected: {expected}")
print(f"Missing: {expected - len(all_files)}")

if expected == len(all_files):
    print("\n✓ All 3,348 files exist!")
else:
    # Find which specific month/batch combinations are missing
    print("\nAnalyzing missing files...")
    existing = set()
    for f in all_files:
        name = os.path.basename(f)
        # Format: m_YYYY_MM_b_BBB.parquet
        parts = name.replace(".parquet", "").split("_")
        # parts = ['m', 'YYYY', 'MM', 'b', 'BBB']
        if len(parts) == 5:
            year = int(parts[1])
            month = int(parts[2])
            batch = int(parts[4])
            existing.add((year, month, batch))

    missing = []
    for year in range(2017, 2026):
        for month in range(1, 13):
            for batch in range(31):
                if (year, month, batch) not in existing:
                    missing.append((year, month, batch))

    print(f"Missing count: {len(missing)}")
    print(f"\nFirst 20 missing:")
    for y, m, b in missing[:20]:
        print(f"  m_{y}_{m:02d}_b_{b:03d}")

    # Analyze by month - which months are incomplete?
    from collections import Counter
    incomplete_months = Counter()
    for y, m, b in missing:
        incomplete_months[f"{y}-{m:02d}"] += 1
    print(f"\nMonths with missing files:")
    for month, count in sorted(incomplete_months.items())[:10]:
        print(f"  {month}: {count}/31 missing")

In [ ]:
# Copy partials to /content/ with auto-retry and remount
import os, shutil, glob, time
from tqdm.auto import tqdm
from google.colab import drive

DRIVE_PARTIAL = "/content/drive/MyDrive/thesisv4/data/raw/partial_batches"
LOCAL_PARTIAL = "/content/partial_batches"

os.makedirs(LOCAL_PARTIAL, exist_ok=True)

def get_missing():
    drive_files = sorted(glob.glob(f"{DRIVE_PARTIAL}/*.parquet"))
    local_names = set(os.listdir(LOCAL_PARTIAL))
    return [f for f in drive_files if os.path.basename(f) not in local_names]

def try_remount():
    try:
        drive.flush_and_unmount()
    except:
        pass
    time.sleep(3)
    drive.mount("/content/drive", force_remount=True)
    time.sleep(5)

max_rounds = 30
for round_num in range(max_rounds):
    missing = get_missing()
    if len(missing) == 0:
        print(f"\n✓ All {len(os.listdir(LOCAL_PARTIAL))} files copied!")
        break

    print(f"\n[Round {round_num+1}] Missing: {len(missing)}")
    copied = 0
    consecutive_failures = 0

    pbar = tqdm(missing, desc=f"R{round_num+1}")
    for f in pbar:
        try:
            shutil.copy(f, LOCAL_PARTIAL)
            copied += 1
            consecutive_failures = 0
        except (OSError, IOError):
            consecutive_failures += 1
            if consecutive_failures >= 5:
                pbar.set_postfix_str("remounting...")
                try_remount()
                consecutive_failures = 0
                break

    print(f"  Copied {copied}")
    if copied == 0:
        print("  No progress. Waiting 60s...")
        time.sleep(60)

# Check disk space
total, used, free = shutil.disk_usage("/content")
print(f"\nDisk: {free/(1024**3):.1f} GB free / {total/(1024**3):.1f} GB total")

In [ ]:
# Override PARTIAL_DIR to use local copy (much faster + no network drops)
PARTIAL_DIR = "/content/partial_batches"
print(f"PARTIAL_DIR = {PARTIAL_DIR}")

# Also override where OUT_CSV goes to be local first (faster write)
LOCAL_OUT_CSV = "/content/full_pixels_10m.csv"
print(f"Will write to {LOCAL_OUT_CSV} first, then copy to Drive")

In [ ]:
import glob, os
import pyarrow.parquet as pq
from tqdm.auto import tqdm

LOCAL_PARTIAL = "/content/partial_batches"
local_files = sorted(glob.glob(f"{LOCAL_PARTIAL}/*.parquet"))
print(f"Checking {len(local_files)} local parquet files...")

corrupted = []
for f in tqdm(local_files, desc="Validating"):
    try:
        pq.read_metadata(f)
    except Exception as e:
        corrupted.append(f)
        os.remove(f)

print(f"\n✓ Valid files: {len(local_files) - len(corrupted)}")
print(f"✗ Corrupted files deleted: {len(corrupted)}")
if corrupted:
    print("\nCorrupted files (deleted):")
    for f in corrupted[:10]:
        print(f"  {os.path.basename(f)}")

In [ ]:
import glob
import gc

partial_files = sorted(glob.glob(os.path.join(PARTIAL_DIR, "*.parquet")))
print(f"Merging {len(partial_files)} local partial files...")

# Write to local first
LOCAL_OUT_CSV = "/content/full_pixels_10m.csv"

if os.path.exists(LOCAL_OUT_CSV):
    os.remove(LOCAL_OUT_CSV)
    print("Removed pre-existing local CSV")

coord_lookup = df_coords.set_index("pixel_id")["pixel_type"].to_dict()

required_cols = ["pixel_id", "pixel_type", "year", "month", "burned"]
feature_cols  = (config.RAW_BANDS + config.INDICES + ["pct_valid"]
                 + config.ANOMALY_BANDS + config.LC_FEATURES)
NON_LC_FEATURES = (config.RAW_BANDS + config.INDICES + ["pct_valid"]
                   + config.ANOMALY_BANDS)

CHUNK = 30  # can be bigger now since reading is fast (local SSD)
total_written = 0
total_burned = 0
total_dropped_by_quality = 0
first_write = True

for i in tqdm(range(0, len(partial_files), CHUNK), desc="Merging"):
    chunk_files = partial_files[i:i+CHUNK]
    dfs = [pd.read_parquet(f) for f in chunk_files]
    df = pd.concat(dfs, ignore_index=True)
    del dfs

    if "burned_this_month" in df.columns:
        df = df.rename(columns={"burned_this_month": "burned"})
    df["pixel_type"] = df["pixel_id"].map(coord_lookup)

    keep_cols = [c for c in required_cols + feature_cols if c in df.columns]
    df = df[keep_cols]
    for c in feature_cols:
        if c not in df.columns:
            df[c] = np.nan
    df[NON_LC_FEATURES] = df[NON_LC_FEATURES].replace(-9999, np.nan)
    df["burned"] = df["burned"].fillna(0).astype(int)

    before = len(df)
    df = df[df["pct_valid"] >= config.SAMPLING["quality_threshold"]]
    total_dropped_by_quality += (before - len(df))

    df.to_csv(LOCAL_OUT_CSV, mode="w" if first_write else "a",
              header=first_write, index=False)
    first_write = False

    total_written += len(df)
    total_burned  += int(df["burned"].sum())
    del df
    gc.collect()

print(f"\n✓ Total rows: {total_written:,}")
print(f"  Dropped by quality: {total_dropped_by_quality:,}")
print(f"  Burned: {total_burned:,} ({100 * total_burned / total_written:.2f}%)")

size_mb = os.path.getsize(LOCAL_OUT_CSV) / (1024**2)
print(f"  Local file size: {size_mb:.1f} MB")

In [ ]:
import shutil, os

# Delete local partial_batches - we have the final CSV, don't need these anymore
LOCAL_PARTIAL = "/content/partial_batches"
if os.path.exists(LOCAL_PARTIAL):
    n = len(os.listdir(LOCAL_PARTIAL))
    print(f"Deleting {n} partial files...")
    shutil.rmtree(LOCAL_PARTIAL)
    print("✓ Deleted")

# Check disk space
total, used, free = shutil.disk_usage("/content")
print(f"\nDisk after cleanup: {free/(1024**3):.1f} GB free / {total/(1024**3):.1f} GB total")

In [ ]:
from google.colab import drive
import shutil, os

# Unmount and remount Drive to flush cache
print("Unmounting Drive to flush FUSE cache...")
try:
    drive.flush_and_unmount()
    print("  ✓ Unmounted")
except Exception as e:
    print(f"  Warning: {e}")

# Check disk space after unmount
total, used, free = shutil.disk_usage("/content")
print(f"\nAfter unmount: {free/(1024**3):.1f} GB free / {total/(1024**3):.1f} GB total")

# Also clear other caches
CACHES = [
    "/root/.cache/pip",
    "/root/.cache/huggingface",
    "/root/.cache/matplotlib",
    "/tmp",
]
for cache in CACHES:
    if os.path.exists(cache):
        try:
            for item in os.listdir(cache):
                path = os.path.join(cache, item)
                try:
                    if os.path.isdir(path):
                        shutil.rmtree(path)
                    else:
                        os.remove(path)
                except:
                    pass
            print(f"  Cleared: {cache}")
        except Exception as e:
            print(f"  {cache}: {e}")

# Final check
total, used, free = shutil.disk_usage("/content")
print(f"\nFinal: {free/(1024**3):.1f} GB free / {total/(1024**3):.1f} GB total")

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
import os

auth.authenticate_user()
service = build('drive', 'v3')

LOCAL_CSV = "/content/full_pixels_10m.csv"

def find_folder(name, parent_id="root"):
    query = f"name='{name}' and mimeType='application/vnd.google-apps.folder' and '{parent_id}' in parents and trashed=false"
    results = service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    return files[0]['id'] if files else None

thesisv4_id = find_folder("thesisv4")
data_id = find_folder("data", thesisv4_id)
raw_id = find_folder("raw", data_id)
print(f"Target folder ID: {raw_id}")

# Delete any existing
query = f"name='full_pixels_10m.csv' and '{raw_id}' in parents and trashed=false"
existing = service.files().list(q=query, fields="files(id)").execute().get('files', [])
for f in existing:
    service.files().delete(fileId=f['id']).execute()
    print(f"Deleted existing file")

# Upload
print(f"\nUploading {os.path.getsize(LOCAL_CSV)/(1024**3):.1f} GB via API...")
file_metadata = {'name': 'full_pixels_10m.csv', 'parents': [raw_id]}
media = MediaFileUpload(LOCAL_CSV, mimetype='text/csv', resumable=True, chunksize=100*1024*1024)

request = service.files().create(body=file_metadata, media_body=media, fields='id')
response = None
while response is None:
    status, response = request.next_chunk()
    if status:
        pct = status.progress() * 100
        print(f"  {pct:.1f}%")

print(f"\n✓ Uploaded! File ID: {response.get('id')}")

## 6. Verify

Quick sanity checks that the final CSV looks right.

In [ ]:
import pandas as pd

OUT_CSV = "/content/drive/MyDrive/thesisv4/data/raw/full_pixels_10m.csv"

# Sample the first 1M rows only (faster than reading 65M)
df_check = pd.read_csv(OUT_CSV, nrows=1_000_000)

print(f"Sample: {len(df_check):,} rows")
print(f"Columns: {len(df_check.columns)}")
print(f"\nColumn list:")
print(df_check.columns.tolist())

print(f"\npixel_type distribution (sample):")
print(df_check["pixel_type"].value_counts())

print(f"\nyear/month coverage (sample):")
print(df_check.groupby(["year", "month"]).size().head(10))

print(f"\nBurned rate (sample): {100 * df_check['burned'].mean():.3f}%")

print(f"\nMissing values by column (sample, %):")
missing = df_check.isna().mean().sort_values(ascending=False) * 100
print(missing.head(15).round(2))

In [ ]:
import shutil, os
DRIVE_PARTIAL = "/content/drive/MyDrive/thesisv4/data/raw/partial_batches"
if os.path.exists(DRIVE_PARTIAL):
    print(f"Deleting {DRIVE_PARTIAL}...")
    shutil.rmtree(DRIVE_PARTIAL)
    print("✓ Done")

## Notes

**If the fetch loop stops partway:** re-run the fetch cell. It resumes from
`partial_batches/` and rebuilds only what is missing.

**If the same (month, batch) fails repeatedly:** lower `PIXELS_PER_REQUEST` to
25,000 for a smaller response. If it still fails, the error text names the limit
that was hit.

**If throughput is low:** raise `N_WORKERS` to 40. The high-volume endpoint accepts
more concurrent requests than the interactive endpoint.

**After a successful run:** `partial_batches/` may be deleted to save space, or kept
as a checkpoint.